# Simulation configuration examples

Run a small batch of simulation episodes from one parameter configuration, plot normalized welfare and every agent's payoff over time, and show network snapshots at selected steps. The final section tests whether the final hub tends to be among the first actors or the first agents to receive a link. `CONFIG["mode"]` selects one of two sources:

- `existing_run`: choose the stored parameter row with the highest final normalized welfare for one population size and policy composition.
- `manual`: use the fixed task configuration defined in this notebook.

In [ ]:
import copy
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
from scipy.stats import binomtest

%matplotlib inline
plt.style.use("science")
set_matplotlib_formats("svg")
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 17,
    "axes.labelsize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "legend.title_fontsize": 13,
    "figure.titlesize": 19,
})

project_root = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "pyproject.toml").is_file() and (path / "tasks").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from the network-creation repository.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from network_creation.core.model import ActionType
from network_creation.core.observability import (
    EpisodeEndEvent,
    EpisodeStartEvent,
    ObservationSampling,
    SimulationObserver,
    StepEvent,
)
from network_creation.core.runtime import create_simulation_runtime
from network_creation.io import read_json
from network_creation.observers.agent_payoff_observer import AgentPayoffObserver
from network_creation.observers.social_welfare_observer import (
    CenteredMeanWelfareObserver,
)
from network_creation.observers.state_history_observer import StateHistoryObserver
from network_creation.plotting.episode_series import plot_episode_series_observer
from network_creation.plotting.graph_history import build_circle_layout
from network_creation.stats.topology_welfare import agent_payoff
from network_creation.time_normalization import normalized_source_times
from tasks.policies import load_policy_results
from tasks.policies.agent_parameters import build_sampled_agent_parameter_plan
from tasks.policies.agent_attributes import build_agent_attribute_plan
from tasks.policies.agents import PolicyAgentBuilder
from tasks.policies.sweep import (
    ParameterSamples,
    build_sweep,
    collect_parameter_specs,
    parse_policy_entries,
    sample_parameters,
)

## Configuration

Choose the parameter source, rerun size, sampling interval, and episode to inspect.

In [ ]:
CONFIG = {
    "mode": "existing_run",  # "existing_run" or "manual"
    "existing_run": {
        "result_store": (
            project_root
            / "artifacts"
            / "simulation_runs"
            / "run-20260729T021448.zarr"
        ),
        "n_agents": 100,
        "composition": 1,  # threshold-link agent count: 0 or 1
        "objective": "centered_mean_welfare.mean",
        "normalize_time_for_selection": True,
    },
    "manual": {
        "sampling": {"log2_budget": 0, "seed": 123},
        "simulator": {
            "seed": 19,
            "budget": {"kind": "raw_simulator_steps", "value": 800},
            "n_episodes": 1,
            "actor_scheduler": {"n_actors": 1},
            "initial_graph": {"kind": "empty"},
        },
        "game": {
            "benefit_to_cost_ratio": {"kind": "fixed", "value": 0.1},
            "distance_discount": {"kind": "fixed", "value": 0.9},
        },
        "population": {
            "n_agents": [30],
            "agent_parameters": {
                "activation_weight": {"kind": "fixed", "value": 1.0}
            },
            "visibility_distance": {"kind": "fixed", "value": 5},
            "policies": [
                {
                    "id": "best_response",
                    "kind": "best-response",
                    "parameters": {
                        "outside_optimism": {"kind": "fixed", "value": 0.43},
                        "exploration": {"kind": "fixed", "value": 0.08},
                    },
                }
            ],
            "policy_proportions": {
                "policy": "best_response",
                "values": [1.0],
            },
        },
    },
    "rerun": {
        "seed": None,  # None keeps the source seed
        "budget": None,  # None keeps the source budget
        "n_episodes": 6,
        "sample_every": 10,
    },
    "analysis": {
        "episode": 0,
        "graph_times": [0, 10, 20, 30],
        "graph_side": 2.0,
    },
}

## Helper Definitions

The next cells separate source resolution, simulation instrumentation, analysis tables, and plotting. Their source is collapsed by default; expand a cell to inspect or edit it.

### Episode trace observer

In [ ]:
class HubFormationObserver(SimulationObserver):
    """Record early roles and the exact final graph for each episode."""

    def __init__(self):
        # Every step is needed because the first link need not occur on a graph-sampling step.
        self.observation_sampling = ObservationSampling.periodic(1)
        self.reset()

    def reset(self):
        self.n_episodes = 0
        self._episodes = []
        self._current = None

    def on_episode_start(self, event: EpisodeStartEvent):
        self._current = {
            "episode": event.episode_index,
            "seed": event.seed,
            "first_actor_step": None,
            "first_actor_ids": (),
            "first_link_step": None,
            "first_link_recipient_ids": (),
        }

    def on_step_end(self, event: StepEvent):
        if self._current is None:
            raise RuntimeError("Received a step before an episode started.")

        actor_ids = tuple(sorted({
            int(action.source)
            for decision in event.decisions
            for action in decision.actions
        }))
        if self._current["first_actor_step"] is None and actor_ids:
            self._current["first_actor_step"] = event.step + 1
            self._current["first_actor_ids"] = actor_ids

        recipient_ids = tuple(sorted({
            int(action.target)
            for decision in event.decisions
            for action in decision.actions
            if action.type in {ActionType.ADD, ActionType.BRIDGE}
            and action.target is not None
        }))
        if self._current["first_link_step"] is None and recipient_ids:
            self._current["first_link_step"] = event.step + 1
            self._current["first_link_recipient_ids"] = recipient_ids

    def on_episode_end(self, event: EpisodeEndEvent):
        if self._current is None:
            raise RuntimeError("Received an episode end before an episode started.")

        final_graph = event.final_state.graph.copy()
        node_ids = np.asarray(final_graph.vs["name"], dtype=int)
        degrees = np.asarray(final_graph.degree(mode="all"), dtype=int)
        hub_ids = tuple(
            int(node_ids[index])
            for index in np.flatnonzero(degrees == degrees.max())
        )
        self._current.update({
            "final_hub_ids": hub_ids,
            "final_hub_degree": int(degrees.max()),
            "final_graph": final_graph,
        })
        self._episodes.append(self._current)
        self._current = None
        self.n_episodes += 1

    def get(self):
        return tuple(self._episodes)


### Parameter source

In [ ]:
def composition_dimension(results):
    dimensions = tuple(
        dim
        for dim in results["x"].dims
        if dim not in {"n_agents", "parameter_row", "step", "observable"}
    )
    if len(dimensions) != 1:
        raise ValueError(f"Expected one policy-composition dimension, found {dimensions}.")
    return dimensions[0]


def select_composition(results, dimension, requested):
    values = np.asarray(results.coords[dimension].values, dtype=float)
    matches = np.flatnonzero(np.isclose(values, requested, rtol=0.0, atol=1e-7))
    if matches.size != 1:
        raise ValueError(
            f"Composition {requested} did not uniquely match {values.tolist()}."
        )
    return results.coords[dimension].values[int(matches[0])].item()


def saved_run_config(results, result_store):
    configured_path = Path(results.attrs["config_path"])
    path = configured_path if configured_path.is_file() else result_store / "run_config.json"
    config = read_json(path)
    if not isinstance(config, dict):
        raise ValueError(f"Expected a JSON object in {path}.")
    return config


def narrow_structure(config, n_agents, composition):
    config = copy.deepcopy(config)
    population = config["population"]
    population["n_agents"] = [int(n_agents)]
    if "policy_proportions" in population:
        population["policy_proportions"]["values"] = [float(composition)]
    elif "policy_counts" in population:
        population["policy_counts"]["values"] = [int(composition)]
    else:
        raise ValueError("Population must define policy_proportions or policy_counts.")
    return config


def existing_run_source(options):
    result_store = Path(options["result_store"])
    objective = str(options["objective"])
    normalize_time_for_selection = bool(options["normalize_time_for_selection"])
    results = load_policy_results(
        result_store,
        stats=(objective,),
        normalize_time=normalize_time_for_selection,
    )
    dimension = composition_dimension(results)
    composition = select_composition(results, dimension, options["composition"])
    if normalize_time_for_selection:
        selection_step = results.coords["step"].values[-1]
    else:
        selection_step = results.coords["last_valid_raw_step"].sel(
            n_agents=int(options["n_agents"])
        ).item()
    final_values = results["x"].sel(
        {
            "observable": objective,
            "n_agents": int(options["n_agents"]),
            dimension: composition,
            "step": selection_step,
        },
        drop=True,
    )
    values = np.asarray(final_values.values, dtype=float)
    finite_positions = np.flatnonzero(np.isfinite(values))
    if not finite_positions.size:
        raise ValueError("No finite welfare values match the requested configuration.")
    best_position = int(finite_positions[np.argmax(values[finite_positions])])
    parameter_row = int(final_values.coords["parameter_row"].values[best_position])
    selector = {
        "n_agents": int(options["n_agents"]),
        dimension: composition,
        "parameter_row": parameter_row,
    }
    theta_row = results["theta"].sel(selector)
    theta = {
        str(name): float(value)
        for name, value in zip(
            theta_row.coords["theta_parameter"].values,
            theta_row.values,
            strict=True,
        )
    }
    run_config = narrow_structure(
        saved_run_config(results, result_store),
        options["n_agents"],
        composition,
    )
    parameter_specs = collect_parameter_specs(run_config)["parameters"]
    parameter_names = tuple(parameter_specs)
    missing = set(parameter_names).difference(theta)
    extra = set(theta).difference({"n_agents", *parameter_names})
    if missing or extra:
        raise ValueError(
            f"Stored parameters do not match the run config: "
            f"missing={sorted(missing)}, extra={sorted(extra)}"
        )
    samples = ParameterSamples(
        parameter_names=parameter_names,
        parameter_values=np.asarray(
            [[theta[name] for name in parameter_names]], dtype=np.float32
        ),
        column_index_by_name={name: index for index, name in enumerate(parameter_names)},
    )
    return {
        "config": run_config,
        "parameter_samples": samples,
        "parameter_row_index": parameter_row,
        "summary": {
            "source": str(result_store.relative_to(project_root)),
            "parameter_row": parameter_row,
            "stored_objective": float(values[best_position]),
            "selection_step": float(selection_step),
            "selection_time_axis": results.attrs.get(
                "time_axis", "raw_simulator_steps"
            ),
            "parameters": theta,
        },
    }


def manual_source(config):
    config = copy.deepcopy(config)
    samples = sample_parameters(collect_parameter_specs(config))
    if samples.parameter_values.shape[0] != 1:
        raise ValueError("Manual mode must resolve to exactly one parameter row.")
    return {
        "config": config,
        "parameter_samples": samples,
        "parameter_row_index": 0,
        "summary": {
            "source": "manual configuration",
            "parameters": {
                name: float(value)
                for name, value in zip(
                    samples.parameter_names,
                    samples.parameter_values[0],
                    strict=True,
                )
            },
        },
    }


def prepare_source(config):
    if config["mode"] == "existing_run":
        return existing_run_source(config["existing_run"])
    if config["mode"] == "manual":
        return manual_source(config["manual"])
    raise ValueError("CONFIG['mode'] must be 'existing_run' or 'manual'.")

### Instrumented rerun

In [ ]:
def run_source(source, rerun):
    config = copy.deepcopy(source["config"])
    simulator_config = config["simulator"]
    simulator_config["n_episodes"] = int(rerun["n_episodes"])
    if rerun["seed"] is not None:
        simulator_config["seed"] = int(rerun["seed"])
    if rerun["budget"] is not None:
        simulator_config["budget"] = copy.deepcopy(rerun["budget"])

    policy_entries = parse_policy_entries(config)
    policy_ids = tuple(policy_id for policy_id, _ in policy_entries)
    input_spec = collect_parameter_specs(config)
    parameter_specs = input_spec["parameters"]
    sweep = build_sweep(
        config,
        source["parameter_samples"],
        parameter_specs=parameter_specs,
        policy_ids=policy_ids,
    )
    if sweep.total_inputs != 1:
        raise ValueError(f"Expected one rerun input, found {sweep.total_inputs}.")
    configuration = sweep.configuration_at(0)
    settings = sweep.resolve(configuration)
    sampled_plan = build_sampled_agent_parameter_plan(
        input_spec["policy_parameters"],
        configuration_column_by_name=sweep.column_index_by_name,
        policy_entries=policy_entries,
    )
    agent_attribute_plan = build_agent_attribute_plan(
        input_spec["agent_attribute_definitions"],
        configuration_column_by_name=sweep.column_index_by_name,
        policy_entries=policy_entries,
        n_agents_values=sweep.n_agents_values,
        n_actors=sweep.n_actors,
    )
    agent_builder = PolicyAgentBuilder(
        policy_entries=policy_entries,
        configuration_names=sweep.configuration_names,
        column_index_by_name=sweep.column_index_by_name,
        sampled_parameters_by_policy_id=sampled_plan.parameters_by_policy_id,
    )
    sampled_agent_parameters = sampled_plan.sample(
        source["parameter_row_index"],
        configuration,
        settings,
        max(sweep.n_agents_values),
        agent_builder.policy_entry_for_node,
    )
    resolved_agent_attributes = agent_attribute_plan.resolve(
        configuration,
        settings,
        parameter_row_index=source["parameter_row_index"],
        max_n_agents=max(sweep.n_agents_values),
        policy_for_node=agent_builder.policy_entry_for_node,
    )
    if resolved_agent_attributes.sampled_values.shape[1]:
        sampled_agent_parameters = np.concatenate(
            (sampled_agent_parameters, resolved_agent_attributes.sampled_values),
            axis=1,
        )
    benefit_to_cost_ratio = float(
        configuration[sweep.column_index_by_name["benefit_to_cost_ratio"]]
    )
    distance_discount = float(
        configuration[sweep.column_index_by_name["distance_discount"]]
    )
    observation_sampling = ObservationSampling.periodic(
        int(rerun["sample_every"])
    )
    welfare_observer = CenteredMeanWelfareObserver(
        n_agents=settings.n_agents,
        observation_sampling=observation_sampling,
        benefit_to_cost_ratio=benefit_to_cost_ratio,
        distance_discount=distance_discount,
        aggregations=("mean", "std", "runs"),
    )
    payoff_observer = AgentPayoffObserver(
        n_agents=settings.n_agents,
        observation_sampling=observation_sampling,
        benefit_to_cost_ratio=benefit_to_cost_ratio,
        distance_discount=distance_discount,
        aggregations=("mean", "std", "runs"),
    )
    graph_observer = StateHistoryObserver(
        observation_sampling=observation_sampling,
        keep_runs=True,
    )
    hub_formation_observer = HubFormationObserver()
    _, _, simulator = create_simulation_runtime(
        simulation_settings=settings,
        build_agent=lambda node_id: agent_builder.build_agent(
            node_id,
            configuration,
            settings,
            sampled_agent_parameters=sampled_agent_parameters,
            activation_weight=float(
                resolved_agent_attributes.activation_weights[node_id]
            ),
        ),
        observers=(
            welfare_observer,
            payoff_observer,
            graph_observer,
            hub_formation_observer,
        ),
        seed=settings.seed,
    )
    simulator.run(n_episodes=settings.n_episodes, reset_observers=True)
    return {
        "settings": settings,
        "configuration": {
            name: float(configuration[index])
            for name, index in sweep.column_index_by_name.items()
        },
        "welfare_observer": welfare_observer,
        "payoff_observer": payoff_observer,
        "graph_result": graph_observer.get(),
        "hub_episodes": hub_formation_observer.get(),
        "activation_weights": resolved_agent_attributes.activation_weights.copy(),
    }

### Analysis tables

In [ ]:
def build_payoff_frames(run):
    """Return one time-by-agent payoff table per episode."""

    payoff_observer = run["payoff_observer"]
    payoff_result = payoff_observer.get()
    payoff_runs = payoff_result.outputs.get("runs")
    if not isinstance(payoff_runs, tuple):
        raise ValueError("Agent payoff observer did not retain per-episode runs.")
    times = normalized_source_times(
        payoff_result.steps,
        n_agents=run["settings"].n_agents,
        n_actors=run["settings"].n_actors,
    )
    return tuple(
        pd.DataFrame(
            values,
            index=pd.Index(times, name="activation_opportunities_per_agent"),
            columns=pd.Index(
                range(run["settings"].n_agents), name="agent"
            ),
        )
        for values in payoff_runs
    )


def build_hub_outcome_tables(run):
    """Build episode details and unique-hub match rates without breaking ties."""

    rows = []
    for trace in run["hub_episodes"]:
        hub_ids = trace["final_hub_ids"]
        unique_hub_id = hub_ids[0] if len(hub_ids) == 1 else None
        first_actor_ids = trace["first_actor_ids"]
        first_recipient_ids = trace["first_link_recipient_ids"]
        first_role_ids = set(first_actor_ids).union(first_recipient_ids)
        rows.append({
            "episode": trace["episode"],
            "final_hub_ids": hub_ids,
            "final_hub_degree": trace["final_hub_degree"],
            "unique_final_hub": unique_hub_id,
            "first_actor_step": trace["first_actor_step"],
            "first_actor_ids": first_actor_ids,
            "first_link_step": trace["first_link_step"],
            "first_link_recipient_ids": first_recipient_ids,
            "hub_was_first_actor": (
                None
                if unique_hub_id is None or not first_actor_ids
                else unique_hub_id in first_actor_ids
            ),
            "hub_was_first_link_recipient": (
                None
                if unique_hub_id is None or not first_recipient_ids
                else unique_hub_id in first_recipient_ids
            ),
            "hub_was_either": (
                None
                if unique_hub_id is None or not first_role_ids
                else unique_hub_id in first_role_ids
            ),
            "any_hub_was_first_actor": (
                None
                if not first_actor_ids
                else bool(set(hub_ids).intersection(first_actor_ids))
            ),
            "any_hub_was_first_link_recipient": (
                None
                if not first_recipient_ids
                else bool(set(hub_ids).intersection(first_recipient_ids))
            ),
            "any_hub_was_either": (
                None
                if not first_role_ids
                else bool(set(hub_ids).intersection(first_role_ids))
            ),
        })

    episode_table = pd.DataFrame(rows).set_index("episode")
    summary_rows = []
    for tie_handling, label, column in (
        ("unique final hub", "acted in the first decision step", "hub_was_first_actor"),
        ("unique final hub", "received a link in the first link step", "hub_was_first_link_recipient"),
        ("unique final hub", "had either early role", "hub_was_either"),
        ("final hub set", "acted in the first decision step", "any_hub_was_first_actor"),
        ("final hub set", "received a link in the first link step", "any_hub_was_first_link_recipient"),
        ("final hub set", "had either early role", "any_hub_was_either"),
    ):
        eligible = episode_table[column].dropna().astype(bool)
        matches = int(eligible.sum())
        n_eligible = int(eligible.size)
        summary_rows.append({
            "tie handling": tie_handling,
            "early role": label,
            "matching episodes": matches,
            "eligible episodes": n_eligible,
            "match share": matches / n_eligible if n_eligible else np.nan,
        })
    summary_table = pd.DataFrame(summary_rows).set_index(
        ["tie handling", "early role"]
    )
    return episode_table, summary_table


def build_final_agent_table(run, episode):
    """Describe every agent on the exact final graph of one episode."""

    trace = run["hub_episodes"][episode]
    graph = trace["final_graph"]
    payoff_observer = run["payoff_observer"]
    payoffs = agent_payoff(
        graph,
        benefit_to_cost_ratio=payoff_observer.benefit_to_cost_ratio,
        distance_discount=payoff_observer.distance_discount,
    )
    node_ids = np.asarray(graph.vs["name"], dtype=int)
    table = pd.DataFrame({
        "agent": node_ids,
        "activation_weight": run["activation_weights"],
        "in_degree": np.asarray(graph.degree(mode="in"), dtype=int),
        "out_degree": np.asarray(graph.degree(mode="out"), dtype=int),
        "total_degree": np.asarray(graph.degree(mode="all"), dtype=int),
        "normalized_payoff": payoffs,
        "is_final_hub": np.isin(node_ids, trace["final_hub_ids"]),
        "acted_first": np.isin(node_ids, trace["first_actor_ids"]),
        "received_first_link": np.isin(
            node_ids, trace["first_link_recipient_ids"]
        ),
    }).set_index("agent")
    return table.sort_values(
        ["total_degree", "in_degree", "normalized_payoff"],
        ascending=False,
        kind="stable",
    )

### Plots

In [ ]:
def plot_hub_match_summary(summary):
    """Plot the three early-role match rates with 95% Wilson intervals."""

    questions = (
        ("acted in the first decision step", "Hub acted first"),
        ("received a link in the first link step", "Hub received first link"),
        ("had either early role", "Hub had either early role"),
    )
    series = (
        ("final hub set", "Any final hub", "#1f4e79", "o", -0.12),
        ("unique final hub", "Unique final hub", "#d97706", "s", 0.12),
    )
    y_positions = np.arange(len(questions), dtype=float)
    with plt.ioff():
        figure, axis = plt.subplots(figsize=(9.0, 4.2), layout="constrained")

    for tie_handling, label, color, marker, offset in series:
        shares = []
        lower_errors = []
        upper_errors = []
        annotations = []
        positions = []
        for question_index, (metric, _) in enumerate(questions):
            row = summary.loc[(tie_handling, metric)]
            matches = int(row["matching episodes"])
            total = int(row["eligible episodes"])
            if not total:
                continue
            share = matches / total
            interval = binomtest(matches, total).proportion_ci(
                confidence_level=0.95, method="wilson"
            )
            lower, upper = float(interval.low), float(interval.high)
            shares.append(share)
            lower_errors.append(max(0.0, share - lower))
            upper_errors.append(max(0.0, upper - share))
            annotations.append(f"{matches}/{total}")
            positions.append(y_positions[question_index] + offset)

        axis.errorbar(
            shares,
            positions,
            xerr=np.asarray([lower_errors, upper_errors]),
            fmt=marker,
            markersize=7,
            capsize=4,
            color=color,
            linewidth=1.8,
            label=label,
        )
        for share, position, annotation in zip(
            shares, positions, annotations, strict=True
        ):
            text_offset = 8 if share < 0.85 else -8
            axis.annotate(
                annotation,
                (share, position),
                xytext=(text_offset, 0),
                textcoords="offset points",
                ha="left" if text_offset > 0 else "right",
                va="center",
                fontsize=10,
            )

    axis.axvline(
        0.5, color="#555555", linestyle="--", linewidth=1.2, label="50% threshold"
    )
    ticks = np.linspace(0.0, 1.0, 6)
    axis.set_xticks(ticks, [f"{tick:.0%}" for tick in ticks])
    axis.set_yticks(y_positions, [label for _, label in questions])
    axis.set(
        xlim=(-0.02, 1.02),
        ylim=(len(questions) - 0.5, -0.5),
        xlabel="Share of eligible episodes (95% Wilson interval)",
        title="Was the final hub an early participant?",
    )
    axis.grid(axis="x", alpha=0.25)
    axis.legend(loc="lower right")
    return figure


def plot_simulation_summary(run, *, episode, times, graph_side):
    result = run["graph_result"]
    steps = result.runs_steps[episode]
    graphs = result.runs_graphs[episode]
    normalized_times = normalized_source_times(
        steps,
        n_agents=run["settings"].n_agents,
        n_actors=run["settings"].n_actors,
    )
    graph_indices = [
        int(np.abs(normalized_times - time).argmin()) for time in times
    ]
    snapshots = [
        (float(normalized_times[index]), graphs[index]) for index in graph_indices
    ]
    trace = run["hub_episodes"][episode]
    final_hub_ids = set(trace["final_hub_ids"])
    first_actor_ids = set(trace["first_actor_ids"])
    first_recipient_ids = set(trace["first_link_recipient_ids"])
    series_height = 3.0
    with plt.ioff():
        figure = plt.figure(
            figsize=(graph_side * len(snapshots), series_height + graph_side + 0.5),
            layout="compressed",
        )
    grid = figure.add_gridspec(
        2,
        len(snapshots),
        height_ratios=(series_height, graph_side),
    )

    welfare_observer = run["welfare_observer"]
    welfare_times = normalized_source_times(
        welfare_observer.steps,
        n_agents=run["settings"].n_agents,
        n_actors=run["settings"].n_actors,
    )
    series_axis = figure.add_subplot(grid[0, :])
    plot_episode_series_observer(
        observer=welfare_observer,
        ax=series_axis,
        show_runs=True,
        x_values=welfare_times,
        x_label="activation opportunities per agent",
        band_color="#1f4e79",
    )
    series_axis.set_title(
        "Normalized welfare across rerun episodes\n"
        f"episode {episode}: final hub(s)={tuple(sorted(final_hub_ids))}, "
        f"first actor(s)={tuple(sorted(first_actor_ids))}, "
        f"first link recipient(s)={tuple(sorted(first_recipient_ids))}"
    )

    with plt.rc_context():
        for column, (time, graph) in enumerate(snapshots):
            axis = figure.add_subplot(grid[1, column])
            graph_nx = graph.to_networkx(vertex_attr_hashable="name")
            nodes = tuple(graph_nx)
            circle = build_circle_layout(len(nodes))
            positions = {
                node: np.asarray(circle[index], dtype=float)
                for index, node in enumerate(nodes)
            }

            nx.draw_networkx_edges(
                graph_nx,
                pos=positions,
                width=0.45,
                edge_color="#1f4e79",
                alpha=0.28,
                arrows=False,
                ax=axis,
            )
            nx.draw_networkx_nodes(
                graph_nx,
                pos=positions,
                node_size=[45 if node in final_hub_ids else 18 for node in nodes],
                node_color=[
                    "#f6bd60" if node in final_hub_ids else "#1f4e79"
                    for node in nodes
                ],
                node_shape="o",
                edgecolors="#18354d",
                linewidths=0.35,
                ax=axis,
            )
            labeled_role_ids = first_actor_ids.union(first_recipient_ids)
            if len(final_hub_ids) <= 3:
                labeled_role_ids.update(final_hub_ids)
            role_labels = {}
            for node in labeled_role_ids.intersection(nodes):
                roles = []
                if node in final_hub_ids:
                    roles.append("H")
                if node in first_actor_ids:
                    roles.append("A")
                if node in first_recipient_ids:
                    roles.append("R")
                role_labels[node] = f"{node} ({'/'.join(roles)})"
            nx.draw_networkx_labels(
                graph_nx,
                pos=positions,
                labels=role_labels,
                font_size=6,
                verticalalignment="bottom",
                ax=axis,
            )
            axis.set(xlim=(-1.08, 1.08), ylim=(-1.08, 1.08), aspect="equal")
            axis.set_axis_off()
            axis.text(
                0.5,
                -0.04,
                f"Time {time:g}",
                transform=axis.transAxes,
                ha="center",
                va="top",
            )
    return figure

## Select and Rerun

Resolve one parameter vector, then rerun it with graph, payoff, welfare, and early-role instrumentation.

In [ ]:
source = prepare_source(CONFIG)
display(source["summary"])

In [ ]:
run = run_source(source, CONFIG["rerun"])
display({
    "n_agents": run["settings"].n_agents,
    "n_actors": run["settings"].n_actors,
    "resolved_raw_steps": run["settings"].max_steps,
    "n_episodes": run["welfare_observer"].n_episodes,
    "resolved_configuration": run["configuration"],
})

## Does an early role predict the final hub?

Definitions used below:

- **Final hub:** the agent with maximum total degree (in-degree + out-degree) on the exact final directed graph.
- **Acts first:** emits a decision in the first decision step; a `PASS` still counts as acting. Actors selected in the same step are tied.
- **Receives a link first:** is the target of an `ADD` or `BRIDGE` action in the earliest step containing a new link. Recipients in the same step are tied.

The figure reports **unique final hub** (excluding tied-hub episodes) and **final hub set** (whether at least one tied hub had the role). Points are match shares, whiskers are 95% Wilson intervals, labels are matches/eligible episodes, and the dashed 50% line marks the literal threshold for "usually." Recipient-only estimates exclude episodes with no new link. Increase `CONFIG["rerun"]["n_episodes"]` for inferential use.

In [ ]:
payoff_by_episode = build_payoff_frames(run)
hub_episode_table, hub_match_summary = build_hub_outcome_tables(run)

display(hub_match_summary.style.format({"match share": "{:.1%}"}, na_rep="n/a"))
hub_match_figure = plot_hub_match_summary(hub_match_summary)
display(hub_match_figure)
plt.close(hub_match_figure)

## Inspect One Episode

Show every agent's exact final payoff and degree, then selected network snapshots. Full time-by-agent payoff histories remain available in `payoff_by_episode[episode]`.

In [ ]:
analysis_episode = int(CONFIG["analysis"]["episode"])
if not 0 <= analysis_episode < len(payoff_by_episode):
    raise IndexError("CONFIG['analysis']['episode'] is outside the rerun batch.")

display(build_final_agent_table(run, analysis_episode))

In [ ]:
analysis_config = CONFIG["analysis"]
summary_figure = plot_simulation_summary(
    run,
    episode=analysis_episode,
    times=analysis_config["graph_times"],
    graph_side=float(analysis_config["graph_side"]),
)
display(summary_figure)
plt.close(summary_figure)